# Collect Data

In [71]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [72]:
import pandas as pd
import numpy as np

In [73]:
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [74]:
df = pd.read_csv('final_data_test.csv')
df['Periodo'] = pd.to_datetime(df['Periodo'], format='%Y-%m-%d')
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776


In [75]:
from sklearn.ensemble import IsolationForest

features = df.drop(columns=['Nome_Poco_ANP', 'Periodo', "Estado", 'Bacia', 'Latitude', 'Longitude', 'Campo'])
features = features.dropna()

iso_forest = IsolationForest(random_state=42)
iso_forest.fit(features)

df = df.loc[features.index].copy()
df['anomaly_score'] = iso_forest.decision_function(features)
df['anomaly'] = iso_forest.predict(features)

df['anomaly'].value_counts()

df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.095040,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.108205,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.107474,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.098098,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.084196,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825,0.048606,1
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033,0.047444,1
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177,0.040722,1
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776,0.046966,1


In [76]:
sample_df = df[df['anomaly'] == 1]
sample_df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.095040,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.108205,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.107474,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.098098,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.084196,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825,0.048606,1
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033,0.047444,1
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177,0.040722,1
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776,0.046966,1


In [77]:
target_column = 'Petroleo'
ignore_columns = [
    'Nome_Poco_ANP', 'Latitude', 'Longitude', 'Campo', 'Estado', 'Bacia', 'Periodo'
]

In [78]:
# sample_df = df.sample(frac=1, random_state=42)
# sample_df.drop_duplicates(inplace=True)
# columns = [col for col in sample_df.columns if col not in ignore_columns]
# for col in columns:
#     Q1 = sample_df[col].quantile(0.25)
#     Q3 = sample_df[col].quantile(0.75)
#     IQR = Q3 - Q1

#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR

#     sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

# sample_df

In [79]:
train = sample_df[sample_df['Periodo'] < pd.to_datetime('2024-07-01')]
test = sample_df[sample_df['Periodo'] >= pd.to_datetime('2024-07-01')]

In [80]:
train

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.095040,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.108205,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.107474,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.098098,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.084196,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825,0.048606,1
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033,0.047444,1
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177,0.040722,1
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,...,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776,0.046966,1


In [81]:
X_train = train.drop([target_column] + ignore_columns, axis=1)
X_test = test.drop([target_column] + ignore_columns, axis=1)

y_train = train[target_column]
y_test = test[target_column]

In [82]:
# train = sample_df[~sample_df['Nome_Poco_ANP'].isin(pocos_teste)]
# test = sample_df[sample_df['Nome_Poco_ANP'].isin(pocos_teste)]

# X_train = train.drop([target_column] + ignore_columns, axis=1)
# y_train = train[target_column]

# X_test = test.drop([target_column] + ignore_columns, axis=1)
# y_test = test[target_column]

In [83]:
# X = sample_df.drop(ignore_columns + [target_column], axis=1)
# y = sample_df[target_column]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.18, random_state=42)

In [84]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [85]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [86]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

# LinearRegression

In [87]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


# Ridge

In [88]:
alphas = np.logspace(-4, 2, 150)

scoring_metric = 'neg_mean_squared_error'
cv=3
ridge_model = Ridge(random_state=42)
grid_search_ridge = GridSearchCV(
    estimator=ridge_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_ridge.fit(X_train_scaled, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['alpha']
best_ridge_model = grid_search_ridge.best_estimator_

Fitting 3 folds for each of 150 candidates, totalling 450 fits


In [89]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,alpha,np.float64(100.0)
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,42


# Lasso

In [90]:
lasso_model = Lasso(random_state=42, max_iter=2000) 
grid_search_lasso = GridSearchCV(
    estimator=lasso_model,
    param_grid={'alpha': alphas},
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_lasso.fit(X_train_scaled, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['alpha']
best_lasso_model = grid_search_lasso.best_estimator_

Fitting 3 folds for each of 150 candidates, totalling 450 fits


In [91]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

,alpha,np.float64(100.0)
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,2000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,42
,selection,'cyclic'


In [92]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.523534,0.656845,2465.430018,2555.237216,1588.583213,1547.912786,0.020023,N/A
1,Ridge Otimo,0.522677,0.655042,2467.646326,2561.938141,1587.195875,1545.530202,0.002880,100.0
2,Lasso Otimo,0.514820,0.647827,2487.872225,2588.592324,1538.853941,1488.894533,0.023695,100.0


# Random Forest

In [93]:
from sklearn.ensemble import RandomForestRegressor

In [94]:
rf_model = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid={
        'n_estimators': [1400, 1450, 1500, 1550, 1600],
        'min_samples_split': range(1, 10, 2),
        'min_samples_leaf': range(1, 10, 2),
        'bootstrap': [True, False],
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_rf.fit(X_train_scaled, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_min_samples_split_rf = grid_search_rf.best_params_['min_samples_split']
best_min_samples_leaf_rf = grid_search_rf.best_params_['min_samples_leaf']
best_bootstrap_rf = grid_search_rf.best_params_['bootstrap']
best_rf_model = grid_search_rf.best_estimator_

Fitting 3 folds for each of 250 candidates, totalling 750 fits


c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
150 fits failed out of a total of 750.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
150 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\PC GAMER\Downloads\Codigos_mestrado\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_tr

In [95]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,n_estimators,1500
,criterion,'squared_error'
,max_depth,None
,min_samples_split,3
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


# Gradient Boosting

In [96]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [97]:
from xgboost import XGBRegressor

In [98]:
xgb_model = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid={
        'n_estimators': [2000, 2500, 3000, 3500, 4000],
        'learning_rate': [.0025, .01, .025, .005],
        'max_depth': [9, 11, 13, 15],
        'subsample': [.7, .75, .8],
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)
grid_search_xgb.fit(X_train_scaled, y_train)

best_n_estimators_xgb = grid_search_xgb.best_params_['n_estimators']
best_max_depth_xgb = grid_search_xgb.best_params_['max_depth']
best_learning_rate_xgb = grid_search_xgb.best_params_['learning_rate']
best_subsample_xgb = grid_search_xgb.best_params_['subsample']
best_xgb_model = grid_search_xgb.best_estimator_

Fitting 3 folds for each of 240 candidates, totalling 720 fits


In [99]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


# KNN Regression

In [100]:
from sklearn.neighbors import KNeighborsRegressor

In [101]:
knn_model = KNeighborsRegressor()
grid_search_knn = GridSearchCV(
    estimator=knn_model,
    param_grid={
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'p': range(1, 6),
        'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
        'leaf_size': [30, 40, 50, 60]
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_knn.fit(X_train_scaled, y_train)
best_n_neighbors_knn = grid_search_knn.best_params_['n_neighbors']
best_weights_knn = grid_search_knn.best_params_['weights']
best_p_knn = grid_search_knn.best_params_['p']
best_algorithm_knn = grid_search_knn.best_params_['algorithm']
best_leaf_size_knn = grid_search_knn.best_params_['leaf_size']
best_knn_model = grid_search_knn.best_estimator_

Fitting 3 folds for each of 800 candidates, totalling 2400 fits


In [102]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,n_neighbors,11
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,1
,metric,'minkowski'
,metric_params,None
,n_jobs,None


# SVR (Suport Vector Regression)

In [103]:
from sklearn.svm import SVR

In [ ]:
svr_model = SVR()
grid_search_svr = GridSearchCV(
    estimator=svr_model,
    param_grid={
        'C': [11000, 12000, 13000, 14000, 15000],
        'epsilon': [.5, 1, 2, 3],
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
        'gamma': ['scale', 'auto'],
        'coef0': [0, .1, .2]
    },
    scoring=scoring_metric,
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search_svr.fit(X_train_scaled, y_train)

best_C_svr = grid_search_svr.best_params_['C']
best_epsilon_svr = grid_search_svr.best_params_['epsilon']
best_kernel_svr = grid_search_svr.best_params_['kernel']
best_gamma_svr = grid_search_svr.best_params_['gamma']
best_coef0_svr = grid_search_svr.best_params_['coef0']
best_svr_model = grid_search_svr.best_estimator_

Fitting 3 folds for each of 480 candidates, totalling 1440 fits


In [ ]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,tol,0.001
,C,11000
,epsilon,1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


# Neural Network

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

In [ ]:
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=42, max_iter=2000))
])

mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(100, 100), (200, 200),
                                (100, 100, 100), (100, 200, 100)],
    'mlp__activation': ['relu', 'identity', 'logistic', 'tanh'],
    'mlp__solver': ['lbfgs', 'sgd', 'adam'],
    'mlp__alpha': [.1, .15, .2],
    'mlp__learning_rate': ['constant', 'invscaling', 'adaptive']
}

grid_search_mlp = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search_mlp.fit(X_train, y_train)
best_params_mlp = grid_search_mlp.best_params_
best_mlp_model = grid_search_mlp.best_estimator_

# GridSearch Details

In [ ]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")
record_gridsearch_details(grid_search_mlp, "Neural Network")

In [ ]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.83864,0.809746,315.185749,275.772423,148.670916,168.5937,0.019505,N/A


In [ ]:
results_df.to_csv('model_comparison_results.csv', index=False)